## Evo2 20B: Fine-Tune from Savanna Checkpoint & Export to Vortex

### Overview

This notebook demonstrates the full lifecycle of working with the Evo2 20B Hyena model:

1. **Download & Convert** the Savanna checkpoint from HuggingFace to MBridge format
2. **Fine-tune** on sample genomic data using tensor parallelism
3. **Export** the fine-tuned MBridge checkpoint to Vortex format for use with ARC's inference code

### Hardware Requirements

The 20B model has ~40GB of bf16 weights and requires multi-GPU setups:

| GPU Type         | Minimum GPUs | Tensor Parallel Size |
| ---------------- | ------------ | -------------------- |
| H100 80GB        | 2            | 2                    |
| A100 80GB        | 2            | 2                    |
| A100/RTX 40GB    | 4            | 4                    |

For `FAST_CI_MODE`, a single GPU with 24GB+ is sufficient (uses a 4-layer subset).

In [3]:
import os
import shutil


FAST_CI_MODE: bool = bool(int(os.environ.get("FAST_CI_MODE", "0")))

CLEANUP: bool = False
if CLEANUP:
    for d in [
        "preprocessed_data",
        "evo2_20b_mbridge",
        "evo2_20b_finetune",
        "evo2_20b_vortex",
        "training_data_config.yaml",
        "preprocess_config.yaml",
    ]:
        if os.path.isdir(d):
            shutil.rmtree(d)
        elif os.path.isfile(d):
            os.remove(d)

### Step 1: Download & Convert Savanna → MBridge

The `evo2_convert_savanna_to_mbridge` CLI downloads the Savanna checkpoint from
HuggingFace and converts it to MBridge (Megatron Bridge) format in one step.
The `--savanna-ckpt-path` flag accepts a HuggingFace repo ID directly.

We pass `--revision` with a pinned commit SHA for reproducibility and security.
The conversion handles layer re-numbering, MLP weight merging, Transformer Engine
layernorm folding, and packaging as a DCP checkpoint with `run_config.yaml`.

In [ ]:
from pathlib import Path

from bionemo.core.utils.subprocess_utils import run_subprocess_safely
from bionemo.evo2.data.dataset_tokenizer import DEFAULT_HF_TOKENIZER_MODEL_PATH_512


SAVANNA_REPO = "arcinstitute/savanna_evo2_20b"
SAVANNA_REVISION = "main"

mbridge_ckpt_path = Path("evo2_20b_mbridge")

if not mbridge_ckpt_path.exists():
    convert_cmd = (
        f"evo2_convert_savanna_to_mbridge "
        f"--savanna-ckpt-path {SAVANNA_REPO} "
        f"--revision {SAVANNA_REVISION} "
        f"--mbridge-ckpt-dir {mbridge_ckpt_path} "
        f"--model-size evo2_20b "
        f"--tokenizer-path {DEFAULT_HF_TOKENIZER_MODEL_PATH_512} "
        f"--mixed-precision-recipe bf16_mixed "
        f"--seq-length 1048576"
    )
    print(f"Downloading & converting savanna → mbridge ...\nCommand: {convert_cmd}")
    result = run_subprocess_safely(convert_cmd)
    if result["returncode"] != 0:
        print(result["stderr"][-3000:])
        raise RuntimeError("Savanna → MBridge conversion failed")
    print(f"MBridge checkpoint created at {mbridge_ckpt_path}")
else:
    print(f"MBridge checkpoint already exists at {mbridge_ckpt_path}")

!ls -la {mbridge_ckpt_path}/

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Command: evo2_convert_savanna_to_mbridge --savanna-ckpt-path arcinstitute/savanna_evo2_20b --revision main --mbridge-ckpt-dir evo2_20b_mbridge --model-size evo2_20b --tokenizer-path /workspaces/bionemo-framework/bionemo-recipes/recipes/evo2_megatron/tokenizers/nucleotide_fast_tokenizer_512 --mixed-precision-recipe bf16_mixed --seq-length 8192


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO:bionemo.evo2.utils.checkpoint.savanna_to_mbridge:Path arcinstitute/savanna_evo2_20b not found locally, treating as HF repo ID...


### Step 2: Prepare Training Data

We reuse the same data preparation pipeline as the 1B fine-tuning tutorial: download
human chromosomes 20, 21, and 22 and preprocess them into Megatron's indexed binary format.

In [ ]:
%%capture

concat_path = "chr20_21_22.fa"
if not os.path.exists(concat_path):
    !wget -q https://hgdownload.soe.ucsc.edu/goldenpath/hg38/chromosomes/chr20.fa.gz
    !wget -q https://hgdownload.soe.ucsc.edu/goldenpath/hg38/chromosomes/chr21.fa.gz
    !wget -q https://hgdownload.soe.ucsc.edu/goldenpath/hg38/chromosomes/chr22.fa.gz
    !zcat chr20.fa.gz > chr20.fa
    !zcat chr21.fa.gz > chr21.fa
    !zcat chr22.fa.gz > chr22.fa
    !cat chr20.fa chr21.fa chr22.fa > chr20_21_22.fa

In [ ]:
full_fasta_path = os.path.abspath(concat_path)
output_dir = os.path.abspath("preprocessed_data")

preprocess_yaml = f"""
- datapaths: ["{full_fasta_path}"]
  output_dir: "{output_dir}"
  output_prefix: chr20_21_22_uint8_distinct
  train_split: 0.9
  valid_split: 0.05
  test_split: 0.05
  overwrite: True
  embed_reverse_complement: true
  random_reverse_complement: 0.0
  random_lineage_dropout: 0.0
  include_sequence_id: false
  transcribe: "back_transcribe"
  force_uppercase: false
  indexed_dataset_dtype: "uint8"
  hf_tokenizer_model_path: {DEFAULT_HF_TOKENIZER_MODEL_PATH_512}
  pretrained_tokenizer_model: null
  special_tokens: null
  fast_hf_tokenizer: true
  append_eod: true
  enforce_sample_length: null
  ftfy: false
  workers: 1
  preproc_concurrency: 100000
  chunksize: 25
  drop_empty_sequences: true
  nnn_filter: false
  seed: 12342
"""
with open("preprocess_config.yaml", "w") as f:
    f.write(preprocess_yaml)

In [ ]:
%%capture
!preprocess_evo2 --config preprocess_config.yaml

In [ ]:
!ls -lh preprocessed_data/

### Step 3: Fine-Tune the 20B Model

We fine-tune using `train_evo2` with the converted MBridge checkpoint.
Key differences from the 1B tutorial:

- **Tensor parallelism** (`--tensor-model-parallel 2`): the 20B model is too large
  for a single GPU. TP=2 splits the model across 2 GPUs.
- **Sequence length 8192**: we use the base context length for the tutorial to keep
  memory manageable. The model supports up to 1M context.
- **Activation checkpointing**: recomputing activations saves memory at the cost of speed.

In `FAST_CI_MODE`, we use a 4-layer subset on a single GPU to verify the pipeline works.

In [ ]:
import torch


output_pfx = str(
    Path(os.path.abspath("preprocessed_data")) / "chr20_21_22_uint8_distinct_nucleotide_fast_tokenizer_512"
)
training_data_yaml = f"""
- dataset_prefix: {output_pfx}_train
  dataset_split: train
  dataset_weight: 1.0
- dataset_prefix: {output_pfx}_val
  dataset_split: validation
  dataset_weight: 1.0
- dataset_prefix: {output_pfx}_test
  dataset_split: test
  dataset_weight: 1.0
"""
with open("training_data_config.yaml", "w") as f:
    f.write(training_data_yaml)

num_gpus = torch.cuda.device_count()

if FAST_CI_MODE:
    MAX_STEPS = 5
    model_subset = "--num-layers 4 --hybrid-override-pattern SDH*"
    tp_size = 1
    actckpt = "--activation-checkpoint-recompute-num-layers 2"
else:
    MAX_STEPS = 20
    model_subset = ""
    tp_size = min(2, num_gpus)
    actckpt = "--activation-checkpoint-recompute-num-layers 5"

result_dir = "evo2_20b_finetune"
val_check_interval = max(1, MAX_STEPS // 2)

train_cmd = (
    f"torchrun --nproc_per_node={num_gpus} --no-python train_evo2 "
    f"-d training_data_config.yaml "
    f"--dataset-dir ./preprocessed_data "
    f"--result-dir {result_dir} "
    f"--experiment-name evo2 "
    f"--model-size evo2_20b "
    f"--seq-length 8192 "
    f"--tensor-model-parallel {tp_size} "
    f"--micro-batch-size 1 "
    f"--global-batch-size 8 "
    f"--eval-iters 5 "
    f"--eval-interval {val_check_interval} "
    f"--hf-tokenizer-model-path {DEFAULT_HF_TOKENIZER_MODEL_PATH_512} "
    f"--lr 0.000015 --min-lr 0.0000149 "
    f"--decay-steps 100000 --warmup-steps 10 "
    f"--max-steps {MAX_STEPS} "
    f"--finetune-ckpt-dir {mbridge_ckpt_path} "
    f"--clip-grad 250 "
    f"--wd 0.001 "
    f"--attention-dropout 0.01 --hidden-dropout 0.01 "
    f"--mixed-precision-recipe bf16_mixed "
    f"{actckpt} {model_subset}"
)

print(f"Training with {num_gpus} GPU(s), TP={tp_size}, {MAX_STEPS} steps")
print(f"Command: {train_cmd}")

In [ ]:
%%capture
result = run_subprocess_safely(train_cmd)

In [ ]:
if result["returncode"] != 0:
    print("================== STDOUT ==========================")
    print(result["stdout"][-5000:])
    print("================== STDERR ==========================")
    print(result["stderr"][-5000:])
    raise AssertionError("Training failed. See output above.")
else:
    print("Training completed successfully!")

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd
import tensorboard.backend.event_processing.event_accumulator as ea


log_dirs = sorted(glob.glob(f"{result_dir}/evo2/tb_logs/events.out.tfevents*"))
if log_dirs:
    acc = ea.EventAccumulator(log_dirs[-1])
    acc.Reload()
    if "lm loss" in acc.Tags()["scalars"]:
        loss_events = acc.Scalars("lm loss")
        steps = [e.step for e in loss_events]
        losses = [e.value for e in loss_events]
        df = pd.DataFrame({"step": steps, "loss": losses})
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(df["step"], df["loss"], marker="o", markersize=3)
        ax.set_xlabel("Step")
        ax.set_ylabel("LM Loss")
        ax.set_title("Evo2 20B Fine-Tuning Loss")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No 'lm loss' scalar found in TensorBoard logs.")
else:
    print(f"No TensorBoard logs found in {result_dir}/evo2/tb_logs/")

### Step 4: Export MBridge → Vortex

Vortex is ARC Institute's inference format for Evo2 Hyena models. The export converts
the MBridge distributed checkpoint into a single `.pt` file that can be loaded by ARC's
[evo2](https://github.com/ArcInstitute/evo2) inference repository.

The exporter handles:
- MLP weight splitting (fused `linear_fc1` → separate `l1`/`l2`)
- Hyena filter pole/residue computation from `p`, `R`, `gamma` parameters
- Embedding weight duplication (tied `embedding_layer` + `unembed`)
- Key renaming from MBridge to Vortex conventions

In [ ]:
ft_ckpt_dir = Path(result_dir) / "evo2" / "checkpoints"
iter_dirs = sorted(ft_ckpt_dir.glob("iter_*")) if ft_ckpt_dir.exists() else []

if not iter_dirs:
    print(f"No checkpoints found in {ft_ckpt_dir}. Using the pre-finetune mbridge checkpoint.")
    export_source = mbridge_ckpt_path
else:
    export_source = iter_dirs[-1]
    print(f"Using fine-tuned checkpoint: {export_source}")

vortex_output = Path("evo2_20b_vortex")
vortex_output.mkdir(parents=True, exist_ok=True)
vortex_pt = vortex_output / "evo2_20b_finetuned.pt"

if FAST_CI_MODE:
    print("Skipping vortex export in FAST_CI_MODE (model subset doesn't match full 20b architecture).")
else:
    export_cmd = (
        f"evo2_export_mbridge_to_vortex "
        f"--mbridge-ckpt-dir {export_source} "
        f"--output-path {vortex_pt} "
        f"--model-size evo2_20b"
    )
    print("Exporting mbridge → vortex ...")
    result = run_subprocess_safely(export_cmd)
    if result["returncode"] != 0:
        print(result["stderr"][-3000:])
        raise RuntimeError("MBridge → Vortex export failed")
    print(f"Vortex checkpoint saved to {vortex_pt} ({vortex_pt.stat().st_size / 1e9:.1f} GB)")

In [ ]:
import torch


if not FAST_CI_MODE and vortex_pt.exists():
    vortex_sd = torch.load(str(vortex_pt), map_location="cpu", weights_only=True)
    print(f"Vortex state dict: {len(vortex_sd)} keys")
    print(f"Sample keys: {list(vortex_sd.keys())[:8]}")
    print(f"Embedding shape: {vortex_sd['embedding_layer.weight'].shape}")
    print(f"Unembed shape:   {vortex_sd['unembed.weight'].shape}")
    print(f"Final norm shape: {vortex_sd['norm.scale'].shape}")
    del vortex_sd

### Step 5: Using the Vortex Checkpoint with ARC's Evo2

The exported `.pt` file is compatible with ARC Institute's
[evo2](https://github.com/ArcInstitute/evo2) inference repository. To use it:

1. **Install ARC's evo2 package** in a separate environment:
   ```bash
   pip install evo2
   ```

2. **Load the model** using the `Evo2` class with a local checkpoint path:
   ```python
   from evo2 import Evo2

   model = Evo2(
       model_name='evo2_20b',
       checkpoint_path='evo2_20b_vortex/evo2_20b_finetuned.pt'
   )
   ```

3. **Generate sequences**:
   ```python
   output = model.generate(
       ['ATCGATCGATCG'],
       n_tokens=500,
       temperature=1.0,
   )
   print(output.sequences[0][:100])
   ```

4. **Score sequences** (log-likelihoods):
   ```python
   logits, _ = model(input_ids)  # input_ids: [batch, seq_len]
   ```

For complete examples, see ARC's notebooks:
- `evo2/notebooks/generation/generation_notebook.ipynb` — sequence generation and alignment
- `evo2/notebooks/brca1/brca1_zero_shot_vep.ipynb` — zero-shot variant effect prediction
- `evo2/notebooks/exon_classifier/exon_classifier.ipynb` — exon boundary classification

### Checkpoint Format Reference

The Vortex `.pt` file contains an `OrderedDict` with these key patterns:

| Key pattern                              | Description              |
| ---------------------------------------- | ------------------------ |
| `embedding_layer.weight`                 | Word embedding           |
| `unembed.weight`                         | Output projection (tied) |
| `blocks.{i}.pre_norm.scale`              | Input layernorm          |
| `blocks.{i}.post_norm.scale`             | Pre-MLP layernorm        |
| `blocks.{i}.filter.*`                    | Hyena filter params      |
| `blocks.{i}.projections.weight`          | Hyena projection         |
| `blocks.{i}.out_filter_dense.*`          | Hyena output projection  |
| `blocks.{i}.inner_mha_cls.Wqkv.*`       | Attention QKV            |
| `blocks.{i}.inner_mha_cls.out_proj.*`    | Attention output         |
| `blocks.{i}.mlp.l1.weight`               | MLP gate                 |
| `blocks.{i}.mlp.l2.weight`               | MLP up-projection        |
| `blocks.{i}.mlp.l3.weight`               | MLP down-projection      |
| `norm.scale`                             | Final layernorm          |

### Summary

This notebook demonstrated the complete Evo2 20B workflow:

| Step | Command / Tool                        | Input                    | Output                   |
| ---- | ------------------------------------- | ------------------------ | ------------------------ |
| 1    | `evo2_convert_savanna_to_mbridge`     | HuggingFace repo + revision | MBridge DCP checkpoint |
| 2    | `preprocess_evo2`                     | FASTA files              | Megatron indexed dataset |
| 3    | `train_evo2 --finetune-ckpt-dir`      | MBridge ckpt + data      | Fine-tuned MBridge ckpt  |
| 4    | `evo2_export_mbridge_to_vortex`       | MBridge ckpt             | Vortex `.pt`             |
| 5    | `Evo2(checkpoint_path=...)`           | Vortex `.pt`             | Inference ready           |

The same pipeline works for all Hyena model sizes (`evo2_1b_base`, `evo2_7b`, `evo2_20b`,
`evo2_40b`, etc.) — just change `--model-size` and adjust GPU/TP settings accordingly.